# Nanoeconomics Simulation — Exploration Notebook

Explore W(M,T,R) dynamics and run quick experiments before using the full Gradio UI.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.wealth import compute_wealth
from src.relational import RelationalConfig, RelationalState, spatial_relational_capital
from src.simulation import SingleCommunityConfig, run_single_community
from src.community import CommunityConfig
from src.outcomes import Outcome

## 1. Spatial density curve S(d)

In [ ]:
sqft_range = np.linspace(10, 3000, 500)
s_vals = [spatial_relational_capital(d) for d in sqft_range]

plt.figure(figsize=(10, 4))
plt.plot(sqft_range, s_vals, color='#2563eb', lw=2)
plt.axvline(250, color='#dc2626', ls='--', label='Peak ~250 sqft/resident')
plt.fill_between([50, 200], [0, 0], [1, 1], alpha=0.08, color='green', label='Dense urban village')
plt.fill_between([200, 500], [0, 0], [1, 1], alpha=0.08, color='blue', label='Walkable neighborhood')
plt.fill_between([500, 1500], [0, 0], [1, 1], alpha=0.08, color='orange', label='Suburban')
plt.xlabel('sqft per resident')
plt.ylabel('S(d) — spatial relational capital')
plt.title('Spatial Density → Relational Capital (Jacobs 1961, Alexander 1977)')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Single Community: High-R vs Low-R comparison

In [ ]:
from src.community import TimeAllocation
from src.relational import RelationalConfig

high_r = CommunityConfig(
    name='High-R',
    population=400, sqft_per_resident=200.0,
    relational_config=RelationalConfig(w_family=0.5, w_religion=0.3, w_spatial=0.2),
    time_allocation=TimeAllocation(production=0.35, family=0.30, religion=0.20, spatial_maintenance=0.08, leisure=0.07),
)

low_r = CommunityConfig(
    name='Low-R',
    population=600, sqft_per_resident=900.0,
    relational_config=RelationalConfig(w_family=0.2, w_religion=0.1, w_spatial=0.7),
    time_allocation=TimeAllocation(production=0.55, family=0.10, religion=0.05, spatial_maintenance=0.15, leisure=0.15),
)

r_high = run_single_community(SingleCommunityConfig(community=high_r, n_paths=200, horizon=30, seed=42))
r_low  = run_single_community(SingleCommunityConfig(community=low_r,  n_paths=200, horizon=30, seed=42))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, result, label, color in [
    (axes[0], r_high, 'High-R (dense, religious)', '#16a34a'),
    (axes[1], r_low,  'Low-R (suburban, secular)', '#dc2626'),
]:
    years = result.paths[0].years
    mean = result.mean_trajectory()
    p25 = result.percentile_trajectory(25)
    p75 = result.percentile_trajectory(75)
    ax.fill_between(years, p25, p75, alpha=0.25, color=color)
    ax.plot(years, mean, color=color, lw=2, label='Mean W')
    ax.axhline(result.w0, color='#9ca3af', ls='--', label='W₀')
    fracs = result.outcome_fractions()
    ax.set_title(f'{label}\nGrew: {fracs.get(Outcome.GREW,0)*100:.0f}% | Collapsed: {fracs.get(Outcome.COLLAPSED,0)*100:.0f}%')
    ax.set_xlabel('Year')
    ax.set_ylabel('W')
    ax.legend()
plt.tight_layout()
plt.show()
print('High-R outcomes:', {o.value: f'{v*100:.0f}%' for o,v in r_high.outcome_fractions().items()})
print('Low-R outcomes: ', {o.value: f'{v*100:.0f}%' for o,v in r_low.outcome_fractions().items()})

## 3. Society simulation quick run

In [ ]:
from src.society import SocietyConfig
from src.animation import run_society_with_frames
import time

cfg = SocietyConfig(n_communities=30, horizon=30, seed=42)
t0 = time.time()
result = run_society_with_frames(cfg)
elapsed = time.time() - t0

years = [f.metrics.year for f in result.frames]
total_w = [f.metrics.total_wealth for f in result.frames]
gini = [f.metrics.gini for f in result.frames]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(years, total_w, color='#2563eb', lw=2)
ax1.set_xlabel('Year'); ax1.set_ylabel('Total wealth'); ax1.set_title('Society Total Wealth')
ax2.plot(years, gini, color='#dc2626', lw=2)
ax2.set_xlabel('Year'); ax2.set_ylabel('Gini'); ax2.set_title('Inequality (Gini)')
ax2.set_ylim(0, 1)
plt.tight_layout()
plt.show()

from collections import Counter
final_counts = Counter(o.value for o in result.final_outcomes)
print(f'Elapsed: {elapsed:.2f}s | Final outcomes: {dict(final_counts)}')